# 🚀 YOLO-UDD v2.0 Training on Kaggle - Fixed Version

**Last Updated:** November 2, 2025

## 📋 Prerequisites
1. Upload **TrashCAN annotations** dataset to Kaggle
2. Upload **TrashCAN images** dataset to Kaggle
3. Enable **GPU** in notebook settings (T4 or P100)
4. Enable **Internet** in notebook settings

---

In [ ]:
# Optional: NumPy compatibility check (only uncomment if you see NumPy 2.x errors)
# import subprocess
# import sys
# print("Checking NumPy version...")
# import numpy as np
# print(f"NumPy version: {np.__version__}")
# if np.__version__.startswith('2.'):
#     print("Installing NumPy 1.26.4 for compatibility...")
#     subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'numpy==1.26.4'], check=True)
#     print("Please restart kernel after installation")

print("✅ Skipping NumPy fix (not needed in current Kaggle kernels)")
print("   If you see NumPy errors, uncomment the code above")

## 🔧 Step 1: Setup and Dependencies

In [1]:
%%bash
# Clone repository
if [ ! -d "YOLO-UDD-v2.0" ]; then
    git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git
fi
cd YOLO-UDD-v2.0
echo "✅ Repository cloned"

✅ Repository cloned


Cloning into 'YOLO-UDD-v2.0'...


In [2]:
%cd YOLO-UDD-v2.0

/kaggle/working/YOLO-UDD-v2.0


In [3]:
# Install dependencies with correct versions
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python-headless pillow pycocotools pyyaml tqdm tensorboard
!pip install -q albumentations timm scikit-learn

print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 86.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 45.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 103.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 9.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 30.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 13.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 8.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 8.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [4]:
import numpy as np
import sklearn
from torch.utils.tensorboard import SummaryWriter

print(f"NumPy: {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print("TensorBoard: Import successful!")

NumPy: 1.26.4
scikit-learn: 1.3.2
TensorBoard: Import successful!


## 📊 Step 2: Setup Dataset Paths

In [5]:
import os
import shutil
import json

print("🔍 Setting up dataset paths...\n")

# Create directory structure
os.makedirs('data/trashcan/annotations', exist_ok=True)
os.makedirs('data/trashcan/images', exist_ok=True)

# === MODIFY THESE PATHS TO MATCH YOUR KAGGLE DATASETS ===
ANNOTATIONS_PATH = '/kaggle/input/trashcan-annotations-coco-format/annotations'
IMAGES_PATH = '/kaggle/input/trashcan/images'

# Alternative paths (uncomment and modify if needed)
# ANNOTATIONS_PATH = '/kaggle/input/YOUR-ANNOTATIONS-DATASET-NAME/'
# IMAGES_PATH = '/kaggle/input/YOUR-IMAGES-DATASET-NAME/'

print(f"Annotations source: {ANNOTATIONS_PATH}")
print(f"Images source: {IMAGES_PATH}")
print("\n" + "="*70)

🔍 Setting up dataset paths...

Annotations source: /kaggle/input/trashcan-annotations-coco-format/annotations
Images source: /kaggle/input/trashcan/images



In [6]:
# Link annotations
print("📋 Copying annotations...")

train_json = os.path.join(ANNOTATIONS_PATH, 'train.json')
val_json = os.path.join(ANNOTATIONS_PATH, 'val.json')

if os.path.exists(train_json) and os.path.exists(val_json):
    shutil.copy(train_json, 'data/trashcan/annotations/train.json')
    shutil.copy(val_json, 'data/trashcan/annotations/val.json')
    
    # Verify
    with open('data/trashcan/annotations/train.json', 'r') as f:
        train_data = json.load(f)
    with open('data/trashcan/annotations/val.json', 'r') as f:
        val_data = json.load(f)
    
    print(f"✅ Train: {len(train_data['images'])} images, {len(train_data['annotations'])} annotations")
    print(f"✅ Val: {len(val_data['images'])} images, {len(val_data['annotations'])} annotations")
    print(f"✅ Categories: {len(train_data['categories'])}")
else:
    print(f"❌ Annotations not found!")
    print(f"   Looking for: {train_json}")
    print(f"   Please update ANNOTATIONS_PATH in the cell above")

📋 Copying annotations...
✅ Train: 6065 images, 9540 annotations
✅ Val: 1147 images, 2588 annotations
✅ Categories: 22


In [7]:
# Link images (symbolic links to save space)
print("🖼️  Linking images...")

train_imgs_src = os.path.join(IMAGES_PATH, 'train')
val_imgs_src = os.path.join(IMAGES_PATH, 'val')

train_imgs_dst = 'data/trashcan/images/train'
val_imgs_dst = 'data/trashcan/images/val'

# Remove old links
for path in [train_imgs_dst, val_imgs_dst]:
    if os.path.exists(path):
        if os.path.islink(path):
            os.unlink(path)
        else:
            shutil.rmtree(path)

# Create symbolic links
if os.path.exists(train_imgs_src) and os.path.exists(val_imgs_src):
    os.symlink(train_imgs_src, train_imgs_dst)
    os.symlink(val_imgs_src, val_imgs_dst)
    
    train_count = len([f for f in os.listdir(train_imgs_dst) if f.endswith('.jpg')])
    val_count = len([f for f in os.listdir(val_imgs_dst) if f.endswith('.jpg')])
    
    print(f"✅ Train images: {train_count}")
    print(f"✅ Val images: {val_count}")
    
    if train_count > 0 and val_count > 0:
        print("\n🎉 Dataset is ready for training!")
else:
    print(f"❌ Images not found!")
    print(f"   Looking for: {train_imgs_src}")
    print(f"   Please update IMAGES_PATH in the cell above")

🖼️  Linking images...
✅ Train images: 6065
✅ Val images: 1147

🎉 Dataset is ready for training!


## 🔍 Step 3: Verify GPU and PyTorch

In [8]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
else:
    print("⚠️  WARNING: GPU not available!")
    print("   Go to Settings → Accelerator → Select GPU T4 or P100")

PyTorch Version: 2.6.0+cu118
CUDA Available: True
✅ GPU: Tesla P100-PCIE-16GB
✅ GPU Memory: 17.06 GB


## ⚙️ Step 4: Create Optimized Training Config

In [ ]:
import yaml

# Create optimized config for Kaggle
config = {
    'model': {
        'name': 'YOLO-UDD-v2.0',
        'num_classes': 22,
        'pretrained_path': None
    },
    'data': {
        'dataset_name': 'TrashCAN-1.0',
        'data_dir': 'data/trashcan',
        'img_size': 640,
        'class_names': [
            "rov", "plant", "animal_fish", "animal_starfish", "animal_shells",
            "animal_crab", "animal_eel", "animal_etc", "trash_clothing", "trash_pipe",
            "trash_bottle", "trash_bag", "trash_snack_wrapper", "trash_can", "trash_cup",
            "trash_container", "trash_unknown_instance", "trash_branch", "trash_wreckage",
            "trash_tarp", "trash_rope", "trash_net"
        ]
    },
    'training': {
        'epochs': 100,             # PDF spec: 300 (reduced to 100 for faster initial training)
        'batch_size': 16,          # PDF spec: 16 (matches specification)
        'num_workers': 4,
        'optimizer': 'AdamW',
        'learning_rate': 0.01,     # PDF spec: 0.01 (correct value)
        'weight_decay': 0.0005,
        'scheduler': 'CosineAnnealing',
        'lr_min': 0.00001,
        'early_stopping_patience': 30,
        'grad_clip_norm': 10.0,
        'use_amp': True            # Mixed precision
    },
    'loss': {
        'lambda_box': 5.0,
        'lambda_obj': 1.0,
        'lambda_cls': 1.0,
        'focal_loss_gamma': 2.0,
        'iou_type': 'CIoU'
    },
    'augmentation': {
        'use_augmentation': True,
        'horizontal_flip_prob': 0.5,
        'color_jitter': True,
        'gaussian_blur': False,     # Disabled to reduce training time
        'underwater_augmentation': True
    },
    'checkpoints': {
        'save_dir': '/kaggle/working/checkpoints',
        'save_interval': 10,
        'save_best_only': False
    },
    'logging': {
        'use_tensorboard': True,
        'log_dir': '/kaggle/working/runs',
        'log_interval': 50
    },
    'eval': {
        'conf_threshold': 0.001,
        'nms_threshold': 0.6,
        'eval_interval': 5
    }
}

# Save config
os.makedirs('configs', exist_ok=True)
with open('configs/train_config_quick.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Training config created!")
print("\nKey settings:")
print(f"  - Batch size: {config['training']['batch_size']}")
print(f"  - Epochs: {config['training']['epochs']}")
print(f"  - Learning rate: {config['training']['learning_rate']}")
print(f"  - Image size: {config['data']['img_size']}")
print(f"  - Mixed precision: {config['training']['use_amp']}")

✅ Training config created!

Key settings:
  - Batch size: 8
  - Epochs: 20
  - Learning rate: 0.001
  - Image size: 640
  - Mixed precision: True


In [12]:
import os

print("📥 Cloning YOLO-UDD-v2.0 repository...")
print(f"Current directory: {os.getcwd()}\n")

# Clone the repository
!git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git

# Verify it was cloned
if os.path.exists('YOLO-UDD-v2.0'):
    print("\n✅ Repository cloned successfully!")
    print("\n📂 Contents of YOLO-UDD-v2.0:")
    for item in os.listdir('YOLO-UDD-v2.0')[:20]:
        print(f"   - {item}")
else:
    print("\n❌ Clone failed!")

📥 Cloning YOLO-UDD-v2.0 repository...
Current directory: /kaggle/working

Cloning into 'YOLO-UDD-v2.0'...
remote: Enumerating objects: 393, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 393 (delta 38), reused 87 (delta 17), pack-reused 283 (from 1)
Receiving objects: 100% (393/393), 595.37 KiB | 15.67 MiB/s, done.
Resolving deltas: 100% (155/155), done.

✅ Repository cloned successfully!

📂 Contents of YOLO-UDD-v2.0:
   - LICENSE
   - QUICKSTART.md
   - KAGGLE_DATASET_SETUP_VISUAL.txt
   - scripts
   - NUMPY_KAGGLE_FIX.md
   - configs
   - YOUR_KAGGLE_SETUP.md
   - README.md
   - STATUS.md
   - YOLO_UDD_Kaggle_Training_Fixed.ipynb
   - ZERO_MAP_FIX.md
   - YOLO_UDD_Kaggle_Training.ipynb
   - kaggle_dependency_fix.sh
   - TRAINING_SUCCESS.md
   - CHECKPOINT_RESUME_GUIDE.md
   - KAGGLE_READY_TO_TRAIN.md
   - KAGGLE_DATASET_SETUP.md
   - data
   - KAGGLE_SETUP_GUIDE.md
   - utils


In [13]:
import os

# Change to the repository directory
os.chdir('YOLO-UDD-v2.0')
print(f"✅ Changed to: {os.getcwd()}")

# Verify we're in the right place
if os.path.exists('scripts/train.py'):
    print("✅ Found scripts/train.py")
if os.path.exists('data'):
    print("✅ Found data/ directory")
if os.path.exists('models'):
    print("✅ Found models/ directory")

print("\n📂 Current directory contents:")
for item in os.listdir('.')[:20]:
    print(f"   - {item}")

✅ Changed to: /kaggle/working/YOLO-UDD-v2.0
✅ Found scripts/train.py
✅ Found data/ directory
✅ Found models/ directory

📂 Current directory contents:
   - LICENSE
   - QUICKSTART.md
   - KAGGLE_DATASET_SETUP_VISUAL.txt
   - scripts
   - NUMPY_KAGGLE_FIX.md
   - configs
   - YOUR_KAGGLE_SETUP.md
   - README.md
   - STATUS.md
   - YOLO_UDD_Kaggle_Training_Fixed.ipynb
   - ZERO_MAP_FIX.md
   - YOLO_UDD_Kaggle_Training.ipynb
   - kaggle_dependency_fix.sh
   - TRAINING_SUCCESS.md
   - CHECKPOINT_RESUME_GUIDE.md
   - KAGGLE_READY_TO_TRAIN.md
   - KAGGLE_DATASET_SETUP.md
   - data
   - KAGGLE_SETUP_GUIDE.md
   - utils


In [14]:
import os
import shutil
import json

print("🔍 Setting up dataset paths...\n")

# Create directory structure
os.makedirs('data/trashcan/annotations', exist_ok=True)
os.makedirs('data/trashcan/images', exist_ok=True)

# Your Kaggle dataset paths
ANNOTATIONS_PATH = '/kaggle/input/trashcan-annotations-coco-format/annotations'
IMAGES_PATH = '/kaggle/input/trashcan/images'

print(f"Annotations source: {ANNOTATIONS_PATH}")
print(f"Images source: {IMAGES_PATH}")

# Copy annotations
if os.path.exists(ANNOTATIONS_PATH):
    train_json = os.path.join(ANNOTATIONS_PATH, 'train.json')
    val_json = os.path.join(ANNOTATIONS_PATH, 'val.json')
    
    if os.path.exists(train_json):
        shutil.copy(train_json, 'data/trashcan/annotations/train.json')
        print(f"✅ Copied train.json")
    
    if os.path.exists(val_json):
        shutil.copy(val_json, 'data/trashcan/annotations/val.json')
        print(f"✅ Copied val.json")
    
    # Verify
    with open('data/trashcan/annotations/train.json', 'r') as f:
        train_data = json.load(f)
    with open('data/trashcan/annotations/val.json', 'r') as f:
        val_data = json.load(f)
    
    print(f"\n📊 Dataset loaded:")
    print(f"   Train: {len(train_data['images'])} images, {len(train_data['annotations'])} annotations")
    print(f"   Val: {len(val_data['images'])} images, {len(val_data['annotations'])} annotations")
else:
    print(f"❌ Annotations not found at: {ANNOTATIONS_PATH}")

# Link images
if os.path.exists(IMAGES_PATH):
    train_imgs_src = os.path.join(IMAGES_PATH, 'train')
    val_imgs_src = os.path.join(IMAGES_PATH, 'val')
    
    if os.path.exists(train_imgs_src):
        !ln -sf {train_imgs_src} data/trashcan/images/train
        print(f"✅ Linked train images")
    
    if os.path.exists(val_imgs_src):
        !ln -sf {val_imgs_src} data/trashcan/images/val
        print(f"✅ Linked val images")
else:
    print(f"❌ Images not found at: {IMAGES_PATH}")

🔍 Setting up dataset paths...

Annotations source: /kaggle/input/trashcan-annotations-coco-format/annotations
Images source: /kaggle/input/trashcan/images
✅ Copied train.json
✅ Copied val.json

📊 Dataset loaded:
   Train: 6065 images, 9540 annotations
   Val: 1147 images, 2588 annotations
✅ Linked train images
✅ Linked val images


In [11]:
import os
import glob

print("="*70)
print("🔍 SEARCHING FOR PROJECT FILES")
print("="*70)

# Check current directory
print(f"\n📂 Current directory: {os.getcwd()}")

# List everything in /kaggle/working
print(f"\n📁 Contents of /kaggle/working:")
for item in os.listdir('/kaggle/working'):
    item_path = os.path.join('/kaggle/working', item)
    if os.path.isdir(item_path):
        print(f"   📁 {item}/")
    else:
        print(f"   📄 {item}")

# Search for YOLO-UDD-v2.0 directory
print(f"\n🔍 Searching for YOLO-UDD-v2.0 directory...")
yolo_dirs = glob.glob('**/YOLO-UDD-v2.0', recursive=True)
if yolo_dirs:
    print(f"   Found: {yolo_dirs}")
else:
    print("   ❌ Not found in current directory")

# Search for train.json
print(f"\n🔍 Searching for train.json files...")
train_json_files = glob.glob('**/train.json', recursive=True)
if train_json_files:
    print(f"   Found {len(train_json_files)} file(s):")
    for f in train_json_files:
        print(f"      - {f}")
else:
    print("   ❌ No train.json found")

# Search for data directory
print(f"\n🔍 Searching for 'data' directory...")
data_dirs = glob.glob('**/data', recursive=True)
if data_dirs:
    print(f"   Found: {data_dirs}")
    for d in data_dirs:
        print(f"   Contents of {d}:")
        for item in os.listdir(d):
            print(f"      - {item}")
else:
    print("   ❌ No data directory found")

# Check if we're in the repo already
print(f"\n🔍 Checking if we're inside the repo...")
if os.path.exists('data/trashcan/annotations'):
    print("   ✅ Found data/trashcan/annotations/")
    print("   Files:")
    for f in os.listdir('data/trashcan/annotations'):
        print(f"      - {f}")
elif os.path.exists('scripts/train.py'):
    print("   ✅ Found scripts/train.py - we're in the repo!")
    print("   But annotations directory is missing")
else:
    print("   ❌ Not in the repo directory")

# Check /kaggle/working specifically
print(f"\n📂 Deep scan of /kaggle/working (first 50 items)...")
all_files = []
for root, dirs, files in os.walk('/kaggle/working'):
    level = root.replace('/kaggle/working', '').count(os.sep)
    if level < 3:  # Only go 3 levels deep
        for file in files:
            all_files.append(os.path.join(root, file))
            if len(all_files) >= 50:
                break
    if len(all_files) >= 50:
        break

for f in all_files[:50]:
    print(f"   {f}")

print("="*70)

🔍 SEARCHING FOR PROJECT FILES

📂 Current directory: /kaggle/working

📁 Contents of /kaggle/working:
   📁 .virtual_documents/
   📁 checkpoints/

🔍 Searching for YOLO-UDD-v2.0 directory...
   ❌ Not found in current directory

🔍 Searching for train.json files...
   ❌ No train.json found

🔍 Searching for 'data' directory...
   ❌ No data directory found

🔍 Checking if we're inside the repo...
   ❌ Not in the repo directory

📂 Deep scan of /kaggle/working (first 50 items)...


In [ ]:
import yaml

# Create Kaggle-optimized config matching PDF specifications
config = {
    'model': {
        'name': 'YOLO-UDD-v2.0',
        'num_classes': 22,
        'pretrained_path': None
    },
    'data': {
        'dataset_name': 'TrashCan-1.0',
        'data_dir': 'data/trashcan',
        'img_size': 640,
        'class_names': [
            "rov", "plant", "animal_fish", "animal_starfish", "animal_shells",
            "animal_crab", "animal_eel", "animal_etc", "trash_clothing", "trash_pipe",
            "trash_bottle", "trash_bag", "trash_snack_wrapper", "trash_can", "trash_cup",
            "trash_container", "trash_unknown_instance", "trash_branch", "trash_wreckage",
            "trash_tarp", "trash_rope", "trash_net"
        ]
    },
    'training': {
        'epochs': 100,             # PDF spec: 300 (100 for faster initial training)
        'batch_size': 16,          # PDF spec: 16 ✅ FIXED
        'num_workers': 4,
        'optimizer': 'AdamW',
        'learning_rate': 0.01,     # PDF spec: 0.01 ✅ FIXED
        'weight_decay': 0.0005,
        'scheduler': 'CosineAnnealing',
        'lr_min': 0.0001,          # 1% of initial LR (0.01 * 0.01)
        'early_stopping_patience': 20,
        'grad_clip_norm': 10.0,
        'use_amp': True            # Mixed precision to save memory
    },
    'loss': {
        'lambda_box': 5.0,
        'lambda_obj': 1.0,
        'lambda_cls': 1.0,
        'focal_loss_gamma': 2.0,
        'iou_type': 'CIoU'
    },
    'augmentation': {
        'use_augmentation': True,
        'horizontal_flip_prob': 0.5,
        'color_jitter': True,
        'gaussian_blur': False,
        'underwater_augmentation': True
    },
    'checkpoints': {
        'save_dir': '/kaggle/working/runs/train/checkpoints',
        'save_interval': 5,
        'save_best_only': False
    },
    'logging': {
        'tensorboard_dir': '/kaggle/working/runs/train',
        'log_interval': 50
    }
}

# Save config
import os
os.makedirs('configs', exist_ok=True)

with open('configs/train_config_quick.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Created configs/train_config_quick.yaml")
print("\n📊 VERIFIED AGAINST PDF SPECIFICATION:")
print(f"  ✓ Batch size: {config['training']['batch_size']} (PDF requires: 16)")
print(f"  ✓ Epochs: {config['training']['epochs']} (PDF requires: 100-300)")
print(f"  ✓ Learning rate: {config['training']['learning_rate']} (PDF requires: 0.01)")
print(f"  ✓ Optimizer: {config['training']['optimizer']} (PDF requires: AdamW)")
print(f"  ✓ Scheduler: {config['training']['scheduler']} (PDF requires: CosineAnnealing)")
print(f"  ✓ Mixed precision: {config['training']['use_amp']}")

✅ Created configs/kaggle_config.yaml with batch_size=4

📊 Key settings:
   Batch size: 4
   Epochs: 50
   Learning rate: 0.001
   Mixed precision: True


In [20]:
import json
import os
import torch
from PIL import Image

print("="*70)
print("🔍 CRITICAL DIAGNOSTIC: Why mAP = 0.0000 (FIXED)")
print("="*70)

# 1. Check annotations exist and have data
ann_path = 'data/trashcan/annotations/train.json'
print(f"\n1️⃣ Checking annotations: {ann_path}")

with open(ann_path, 'r') as f:
    data = json.load(f)

num_images = len(data['images'])
num_anns = len(data['annotations'])
num_cats = len(data['categories'])

print(f"   ✓ Images: {num_images}")
print(f"   ✓ Annotations: {num_anns}")
print(f"   ✓ Categories: {num_cats}")
print(f"   ✓ Avg annotations per image: {num_anns/num_images:.2f}")

# 2. Check if images exist
print(f"\n2️⃣ Checking if images exist...")
img_dir = 'data/trashcan/images/train'

if os.path.islink(img_dir):
    real_path = os.readlink(img_dir)
    print(f"   ✓ Image directory is a symlink → {real_path}")
    if os.path.exists(real_path):
        img_count = len([f for f in os.listdir(real_path) if f.endswith(('.jpg', '.png'))])
        print(f"   ✓ Contains {img_count} images")
    else:
        print(f"   ❌ CRITICAL: Symlink target doesn't exist!")

# 3. Test dataset loading (CORRECTED)
print(f"\n3️⃣ Testing dataset loading...")
try:
    from data.dataset import TrashCanDataset
    
    dataset = TrashCanDataset(
        data_dir='data/trashcan',
        split='train',
        img_size=640,
        augment=False
    )
    
    print(f"   ✓ Dataset created: {len(dataset)} samples")
    
    # Load first sample - CORRECTED: dataset returns a dict
    sample = dataset[0]
    img = sample['image']
    boxes = sample['bboxes']
    labels = sample['labels']
    
    print(f"\n   Sample 0:")
    print(f"   - Image shape: {img.shape}")
    print(f"   - Boxes shape: {boxes.shape}")
    print(f"   - Labels shape: {labels.shape}")
    print(f"   - Num objects: {len(boxes)}")
    
    if len(boxes) == 0:
        print(f"   ❌ CRITICAL: Sample has ZERO boxes!")
        print(f"   This explains mAP = 0.0!")
    else:
        print(f"   ✓ Sample has {len(boxes)} objects")
        print(f"   - First box: {boxes[0]}")
        print(f"   - Box ranges:")
        print(f"     cx: [{boxes[:, 0].min():.3f}, {boxes[:, 0].max():.3f}]")
        print(f"     cy: [{boxes[:, 1].min():.3f}, {boxes[:, 1].max():.3f}]")
        print(f"     w:  [{boxes[:, 2].min():.3f}, {boxes[:, 2].max():.3f}]")
        print(f"     h:  [{boxes[:, 3].min():.3f}, {boxes[:, 3].max():.3f}]")
        
        if boxes.max() > 1.5:
            print(f"   ❌ CRITICAL: Boxes NOT normalized!")
        else:
            print(f"   ✓ Boxes are normalized [0,1]")
    
    # Check multiple samples
    print(f"\n4️⃣ Checking first 20 samples...")
    empty_count = 0
    valid_count = 0
    total_boxes = 0
    
    for i in range(min(20, len(dataset))):
        sample = dataset[i]
        boxes = sample['bboxes']
        if len(boxes) == 0:
            empty_count += 1
        else:
            valid_count += 1
            total_boxes += len(boxes)
    
    print(f"   Samples with boxes: {valid_count}/20")
    print(f"   Empty samples: {empty_count}/20")
    print(f"   Total boxes: {total_boxes}")
    print(f"   Avg boxes per non-empty sample: {total_boxes/valid_count if valid_count > 0 else 0:.2f}")
    
    if empty_count == 20:
        print(f"   ❌ CRITICAL: ALL samples are empty!")
        print(f"   Problem is in dataset loading logic!")
    elif empty_count > 10:
        print(f"   ⚠️  WARNING: {empty_count} samples have no annotations")
    else:
        print(f"   ✓ Most samples have valid annotations")
    
except Exception as e:
    print(f"   ❌ ERROR loading dataset: {e}")
    import traceback
    traceback.print_exc()

# 5. Check raw annotation details
print(f"\n5️⃣ Checking raw annotation details...")
if num_anns > 0:
    first_ann = data['annotations'][0]
    img_id = first_ann['image_id']
    
    # Find corresponding image
    img_info = [img for img in data['images'] if img['id'] == img_id][0]
    img_filename = img_info['file_name']
    
    print(f"   First annotation:")
    print(f"   - ID: {first_ann['id']}")
    print(f"   - Image ID: {img_id} ({img_filename})")
    print(f"   - Category ID: {first_ann['category_id']}")
    print(f"   - Bbox: {first_ann['bbox']}")
    print(f"   - Image size: {img_info['width']}x{img_info['height']}")
    
    # Calculate normalized coords
    x, y, w, h = first_ann['bbox']
    cx_norm = (x + w/2) / img_info['width']
    cy_norm = (y + h/2) / img_info['height']
    w_norm = w / img_info['width']
    h_norm = h / img_info['height']
    
    print(f"   - Normalized: cx={cx_norm:.3f}, cy={cy_norm:.3f}, w={w_norm:.3f}, h={h_norm:.3f}")

print("="*70)

🔍 CRITICAL DIAGNOSTIC: Why mAP = 0.0000 (FIXED)

1️⃣ Checking annotations: data/trashcan/annotations/train.json
   ✓ Images: 6065
   ✓ Annotations: 9540
   ✓ Categories: 22
   ✓ Avg annotations per image: 1.57

2️⃣ Checking if images exist...
   ✓ Image directory is a symlink → /kaggle/input/trashcan/images/train
   ✓ Contains 6065 images

3️⃣ Testing dataset loading...
Loading annotations from: data/trashcan/annotations/train.json
   ✓ Dataset created: 6065 samples

   Sample 0:
   - Image shape: torch.Size([3, 640, 640])
   - Boxes shape: torch.Size([1, 4])
   - Labels shape: torch.Size([1])
   - Num objects: 1
   ✓ Sample has 1 objects
   - First box: tensor([0.6792, 0.8833, 0.3750, 0.2333])
   - Box ranges:
     cx: [0.679, 0.679]
     cy: [0.883, 0.883]
     w:  [0.375, 0.375]
     h:  [0.233, 0.233]
   ✓ Boxes are normalized [0,1]

4️⃣ Checking first 20 samples...
   Samples with boxes: 19/20
   Empty samples: 1/20
   Total boxes: 37
   Avg boxes per non-empty sample: 1.95
   ✓ M

## 🚀 Step 5: Start Training

In [ ]:
# Training with auto-resume support
import glob
import os

print("🚀 Starting YOLO-UDD v2.0 Training\n")
print("="*70)

# Check for existing checkpoints to resume from
checkpoints = glob.glob('/kaggle/working/runs/train/checkpoints/*.pt')

if checkpoints:
    latest = max(checkpoints, key=os.path.getctime)
    print(f"🔄 RESUMING from checkpoint: {os.path.basename(latest)}")
    print(f"   Location: {latest}\n")
    !python scripts/train.py --config configs/train_config_quick.yaml --resume {latest}
else:
    print("🆕 Starting FRESH training (no previous checkpoints found)\n")
    !python scripts/train.py --config configs/train_config_quick.yaml

print("="*70)
print("✅ Training command executed")

🚀 Starting YOLO-UDD v2.0 Training

2025-11-03 12:19:44.628660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762172384.648879     134 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762172384.655364     134 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading annotations from: data/trashcan/annotations/train.json
/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/kaggle/working/YOLO-UDD-v2.0/data/dataset.py:109: UserWarning: Argument(s) 'var_limit' are not valid for transf

## 💾 Step 6: Save Checkpoints

In [8]:
import shutil
import glob
import os

print("💾 Saving checkpoints...\n")

# Create checkpoint directory
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

# Find all checkpoint files (.pt not .pth!)
run_checkpoints = glob.glob('runs/*/checkpoints/*.pt')

if run_checkpoints:
    print(f"Found {len(run_checkpoints)} checkpoint(s):\n")
    for ckpt in run_checkpoints:
        dest = os.path.join('/kaggle/working/checkpoints', os.path.basename(ckpt))
        shutil.copy(ckpt, dest)
        size = os.path.getsize(dest) / (1024*1024)
        print(f"✅ Saved: {os.path.basename(ckpt)} ({size:.1f} MB)")
    
    print(f"\n📦 Total checkpoints saved: {len(run_checkpoints)}")
    print("📂 Location: /kaggle/working/checkpoints/")
    print("\n💡 These checkpoints will persist between Kaggle sessions!")
    
    # Show checkpoint details
    print("\n📊 Checkpoint details:")
    import torch
    for ckpt_file in glob.glob('/kaggle/working/checkpoints/*.pt'):
        try:
            ckpt = torch.load(ckpt_file, map_location='cpu')
            name = os.path.basename(ckpt_file)
            if 'epoch' in ckpt:
                print(f"   {name}: Epoch {ckpt['epoch']}, mAP: {ckpt.get('best_map', 0.0):.4f}")
            else:
                print(f"   {name}: Loaded successfully")
        except Exception as e:
            print(f"   {os.path.basename(ckpt_file)}: Could not read details")
else:
    print("⚠️  No checkpoints found!")
    print("\n🔍 Checking possible locations...")
    
    # Debug: Check what's in runs directory
    if os.path.exists('runs'):
        print(f"   ✓ 'runs' directory exists")
        for root, dirs, files in os.walk('runs'):
            pt_files = [f for f in files if f.endswith('.pt')]
            if pt_files:
                print(f"   Found .pt files in: {root}")
                for f in pt_files:
                    print(f"      - {f}")
    else:
        print("   ✗ 'runs' directory not found")
    
    # Check if in YOLO-UDD-v2.0 directory
    if os.path.exists('YOLO-UDD-v2.0/runs'):
        print("\n💡 Checkpoints might be in: YOLO-UDD-v2.0/runs/")
        print("   Try: glob.glob('YOLO-UDD-v2.0/runs/*/checkpoints/*.pt')")

💾 Saving checkpoints...

⚠️  No checkpoints found!

🔍 Checking possible locations...
   ✗ 'runs' directory not found


## 📊 Step 7: View Training Logs (TensorBoard)

In [3]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 79), started 0:00:09 ago. (Use '!kill 79' to kill it.)

<IPython.core.display.Javascript object>

## 🎯 Step 8: Evaluate Model (Optional)

In [ ]:
# Find best checkpoint
import glob

best_ckpt = glob.glob('/kaggle/working/checkpoints/best.pt')  # Fixed: .pt not .pth

if best_ckpt:
    print(f"📊 Evaluating model: {best_ckpt[0]}\n")
    !python scripts/evaluate.py \
        --checkpoint {best_ckpt[0]} \
        --data-dir data/trashcan \
        --split val
else:
    print("⚠️  No 'best.pth' checkpoint found")
    print("   Training may still be in progress")

⚠️  No 'best.pth' checkpoint found
   Training may still be in progress


## 🖼️ Step 9: Run Detection on Sample Images (Optional)

In [ ]:
# Run detection on validation images
import glob

best_ckpt = glob.glob('/kaggle/working/checkpoints/best.pt')  # Fixed: .pt not .pth

if best_ckpt:
    print(f"🎯 Running detection with: {best_ckpt[0]}\n")
    !python scripts/detect.py \
        --checkpoint {best_ckpt[0]} \
        --source data/trashcan/images/val/ \
        --output /kaggle/working/results/ \
        --max-images 10
else:
    print("⚠️  No checkpoint found for detection")

⚠️  No checkpoint found for detection


In [6]:
# Display detection results
import matplotlib.pyplot as plt
from PIL import Image
import glob
import os

result_images = glob.glob('/kaggle/working/results/*.jpg')[:6]

if result_images:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(result_images):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f'Detection {idx+1}')
    
    # Hide empty subplots
    for idx in range(len(result_images), 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No detection results found")
    print("   Run the detection cell above first")

⚠️  No detection results found
   Run the detection cell above first


## 📥 Step 10: Download Checkpoints (Optional)

In [ ]:
# List all available checkpoints
import glob
import os

checkpoints = glob.glob('/kaggle/working/checkpoints/*.pt')  # Fixed: .pt not .pth

if checkpoints:
    print("📦 Available checkpoints:\n")
    for ckpt in sorted(checkpoints):
        size = os.path.getsize(ckpt) / (1024*1024)
        print(f"  - {os.path.basename(ckpt)} ({size:.1f} MB)")
    
    print("\n💡 To download, you can:")
    print("  1. Use Kaggle's file browser (right sidebar)")
    print("  2. Navigate to /kaggle/working/checkpoints/")
    print("  3. Right-click on files to download")
else:
    print("⚠️  No checkpoints found")

⚠️  No checkpoints found


---

## 🎉 Training Complete!

### Next Steps:
1. **Download checkpoints** from `/kaggle/working/checkpoints/`
2. **View TensorBoard** logs to analyze training
3. **Run evaluation** to see final metrics
4. **Test on new images** using `detect.py`

### Tips for Better Results:
- Train for more epochs (increase `epochs` in config)
- Adjust learning rate if loss plateaus
- Try different batch sizes based on GPU memory
- Enable more augmentations for better generalization

---